# XBRL validation (Part 11)

Every number on the statement tables is graded against the filing's own XBRL (issues #39, #57).

**Inputs** (produced by the pipeline, not edited here):
- `data/xbrl/comparison_{path}.csv`: one row per table cell, from `python src/xbrl.py compare --path {path} --tables <folder>`
- `data/xbrl/facts.csv`: Arelle facts, from `python src/xbrl.py --input data/rendered/manifest.csv --output data/xbrl`

**Rerun** from the repo root (executes every cell and saves the outputs):

```
python -c "import nbformat; from nbclient import NotebookClient; p='notebooks/xbrl_validation.ipynb'; nb=nbformat.read(p, 4); NotebookClient(nb, timeout=180, kernel_name='python3', resources={'metadata': {'path': 'notebooks'}}).execute(); nbformat.write(nb, p)"
```

Strict match rate counts `sign` separately, as the case study defines it; value agreement counts `match` + `sign`.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
XBRL = ROOT / "data" / "xbrl"

paths = {f.stem.split("_", 1)[1]: f for f in sorted(XBRL.glob("comparison_*.csv"))}
cmp = pd.concat([pd.read_csv(f).assign(path=name) for name, f in paths.items()], ignore_index=True)
cmp["cause"] = cmp["cause"].fillna("")
cmp["filing"] = cmp.stem.str.split("_").str[1]
facts = pd.read_csv(XBRL / "facts.csv")

print("comparison files:", {name: f.name for name, f in paths.items()})
if "docling" not in paths:
    print("comparison_docling.csv not found yet: run issue #49 (compare --path docling), then rerun this notebook.")
print(cmp.groupby("path").size().rename("cells").to_string())

comparison files: {'traditional': 'comparison_traditional.csv'}
comparison_docling.csv not found yet: run issue #49 (compare --path docling), then rerun this notebook.
path
traditional    388


## 1. Match rate per extraction path and statement

In [2]:
def rates(df, by):
    g = df.groupby(by)
    out = pd.DataFrame({
        "cells": g.size(),
        "match": g.status.apply(lambda s: (s == "match").sum()),
        "sign": g.status.apply(lambda s: (s == "sign").sum()),
    })
    out["other"] = out.cells - out.match - out.sign
    out["match_rate"] = (out.match / out.cells).round(4)
    out["value_agreement"] = ((out.match + out.sign) / out.cells).round(4)
    return out

by_statement = rates(cmp, ["path", "statement"])
overall = rates(cmp, ["path"])
overall.index = pd.MultiIndex.from_tuples([(p, "ALL") for p in overall.index], names=["path", "statement"])
pd.concat([by_statement, overall]).sort_index()

cells  match  sign  other  match_rate  value_agreement
path        statement                                                        
traditional ALL          388    328    60      0      0.8454              1.0
            balance      110    110     0      0      1.0000              1.0
            cash_flow    145     85    60      0      0.5862              1.0
            income       133    133     0      0      1.0000              1.0

## 2. Per filing and statement

In [3]:
rates(cmp, ["path", "filing", "statement"])

cells  match  sign  other  match_rate  value_agreement
path        filing statement                                                        
traditional 10K    balance       54     54     0      0      1.0000              1.0
                   cash_flow     87     51    36      0      0.5862              1.0
                   income        57     57     0      0      1.0000              1.0
            10Q    balance       56     56     0      0      1.0000              1.0
                   cash_flow     58     34    24      0      0.5862              1.0
                   income        76     76     0      0      1.0000              1.0

## 3. Statuses and mapping methods

In [4]:
print(pd.crosstab([cmp.path, cmp.statement], cmp.status, margins=True).to_string())
print()
print(pd.crosstab(cmp.path, cmp.mapping, margins=True).to_string())

status                 match  sign  All
path        statement                  
traditional balance      110     0  110
            cash_flow     85    60  145
            income       133     0  133
All                      328    60  388

mapping      label  manual  All
path                           
traditional     73     315  388
All             73     315  388


## 4. Every non-match, grouped by line, with its cause

Acceptance (#57): every non-match has a diagnosed cause. `sign` rows get their cause automatically when the
filing's presentation linkbase (`_pre.xml`) marks the concept with a negated label. Any row listed under
"undiagnosed" still needs a cause in `reports/xbrl.md`.

In [5]:
nonmatch = cmp[cmp.status != "match"]
lines = (nonmatch.groupby(["path", "filing", "statement", "pdf_label", "concept", "status", "cause"], dropna=False)
         .agg(cells=("status", "size"), periods=("period_label", lambda s: ", ".join(sorted(s))))
         .reset_index())
print(f"{len(nonmatch)} non-match cells on {len(lines)} lines")
lines.drop(columns="periods")

60 non-match cells on 24 lines


,path,filing,statement,pdf_label,concept,status,cause,cells
0,traditional,10K,cash_flow,Adjustments to reconcile net income to cash generated by operating...,OtherNoncashIncomeExpense,sign,presentation: negated label in _pre.xml,3
1,traditional,10K,cash_flow,"Changes in operating assets and liabilities: Accounts receivable, net",IncreaseDecreaseInAccountsReceivable,sign,presentation: negated label in _pre.xml,3
2,traditional,10K,cash_flow,Changes in operating assets and liabilities: Inventories,IncreaseDecreaseInInventories,sign,presentation: negated label in _pre.xml,3
3,traditional,10K,cash_flow,Changes in operating assets and liabilities: Other current and non...,IncreaseDecreaseInOtherOperatingAssets,sign,presentation: negated label in _pre.xml,3
4,traditional,10K,cash_flow,Changes in operating assets and liabilities: Vendor non-trade rece...,IncreaseDecreaseInOtherReceivables,sign,presentation: negated label in _pre.xml,3
5,traditional,10K,cash_flow,Financing activities: Payments for dividends and dividend equivalents,PaymentsOfDividends,sign,presentation: negated label in _pre.xml,3
6,traditional,10K,cash_flow,Financing activities: Payments for taxes related to net share sett...,PaymentsRelatedToTaxWithholdingForShareBasedCompensation,sign,presentation: negated label in _pre.xml,3
7,traditional,10K,cash_flow,Financing activities: Repayments of term debt,RepaymentsOfLongTermDebt,sign,presentation: negated label in _pre.xml,3
8,traditional,10K,cash_flow,Financing activities: Repurchases of common stock,PaymentsForRepurchaseOfCommonStock,sign,presentation: negated label in _pre.xml,3
9,traditional,10K,cash_flow,Investing activities: Other,PaymentsForProceedsFromOtherInvestingActivities,sign,presentation: negated label in _pre.xml,3


In [6]:
undiagnosed = nonmatch[nonmatch.cause == ""]
print("non-match cells without a cause:", len(undiagnosed))
undiagnosed[["path", "stem", "pdf_label", "period_label", "status", "pdf_value", "xbrl_value"]]

non-match cells without a cause: 0


,path,stem,pdf_label,period_label,status,pdf_value,xbrl_value


## 5. What broke and how it was fixed (traditional path)

Recorded from the runs on the real filings during #39 (PR #98; details in `docs/ai_log/dhruvi.md`).
The first run covered the 10-K income and balance sheet and all three 10-Q statements (301 cells);
the 10-K cash flow (p36) was added after the fixes.

| # | Cells | Status before | Cause | Fix |
|---|---|---|---|---|
| 1 | 2 | mismatch | mapping: Intangible assets mapped to the total (`IntangibleAssetsNetExcludingGoodwill`, 25,417M) instead of the non-current part (20,342M) | correct concept in `config/label_map.yaml` |
| 2 | 2 | xbrl_missing | extension concept: `aapl:IntangibleAssetsNetExcludingGoodwillNoncurrent` is Apple's own, but map lines assumed `us-gaap` | optional `prefix` per map line |
| 3 | 2 | mismatch | mapping: the generic label "Other" matched the investing concept on the operating line | three "Other" lines in the manual map; `ambiguous` instead of guessing |
| 4 | 4 | xbrl_missing | period alignment: beginning/ending cash balances are instants in "9M ended" columns | duration columns also try their end and opening instants |
| 5 | 22 | sign | presentation convention, not an error (negated labels in `_pre.xml`) | cause filled automatically |

First run: 271 / 301 match. After the fixes: 277 match + 24 sign of the same 301 cells. With the 10-K cash flow added,
the totals are in section 1.

## 6. One number end to end: net income, latest fiscal year (10-K)

In [7]:
ni = cmp[(cmp.path == "traditional") & (cmp.filing == "10K") & (cmp.statement == "income")
         & (cmp.pdf_label == "Net income")].sort_values("period_label")
row = ni[ni.period_label.str.startswith("FY")].iloc[-1]
fact = facts[(facts.stem == row.stem) & (facts.prefix == row.prefix) & (facts.concept == row.concept)
             & (facts.period_label == row.xbrl_period) & (facts.n_dims == 0)].iloc[0]
end_to_end = pd.DataFrame({
    "PDF (table CSV)": [f"page {row.page}", row.pdf_label, row.period_label, row.pdf_raw, f"{row.pdf_value:,.0f}"],
    "XBRL (Arelle)": [fact.accession, f"{fact.prefix}:{fact.concept}", f"{fact.start} to {fact.end}",
                      f"decimals {fact.decimals}", f"{fact.value:,.0f} {fact.unit}"],
}, index=["where", "what", "period", "raw / precision", "value"])
print("status:", row.status, "| mapping:", row.mapping, "| tolerance:", row.tolerance)
end_to_end

status: match | mapping: manual | tolerance: 500000.0


,PDF (table CSV),XBRL (Arelle)
where,page 32,0000320193-25-000079
what,Net income,us-gaap:NetIncomeLoss
period,FY ended 2025-09-27,2024-09-29 to 2025-09-27
raw / precision,"112,010",decimals -6.0
value,"112,010,000,000","112,010,000,000 usd"


## 7. Label-layer mappings (layer 2: the filing's own `_lab.xml`)

Lines not in the curated map were resolved from Apple's label linkbase. Every one is checked here by value.

In [8]:
lab = cmp[cmp.mapping == "label"]
(lab.groupby(["path", "pdf_label", "concept"])
    .agg(cells=("status", "size"), match=("status", lambda s: (s == "match").sum()),
         sign=("status", lambda s: (s == "sign").sum()))
    .reset_index())

,path,pdf_label,concept,cells,match,sign
0,traditional,Changes in operating assets and liabilities: Accounts payable,IncreaseDecreaseInAccountsPayable,5,5,0
1,traditional,"Changes in operating assets and liabilities: Accounts receivable, net",IncreaseDecreaseInAccountsReceivable,5,0,5
2,traditional,Changes in operating assets and liabilities: Inventories,IncreaseDecreaseInInventories,5,0,5
3,traditional,Changes in operating assets and liabilities: Other current and non...,IncreaseDecreaseInOtherOperatingAssets,5,0,5
4,traditional,Changes in operating assets and liabilities: Other current and non...,IncreaseDecreaseInOtherOperatingLiabilities,5,5,0
5,traditional,Changes in operating assets and liabilities: Vendor non-trade rece...,IncreaseDecreaseInOtherReceivables,5,0,5
6,traditional,"Financing activities: Increase in cash, cash equivalents, and rest...",CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsPerio...,2,2,0
7,traditional,"Financing activities: Increase/(Decrease) in cash, cash equivalent...",CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsPerio...,3,3,0
8,traditional,Financing activities: Payments for taxes related to net share sett...,PaymentsRelatedToTaxWithholdingForShareBasedCompensation,5,0,5
9,traditional,"Financing activities: Proceeds from issuance of term debt, net",ProceedsFromIssuanceOfLongTermDebt,5,5,0


In [9]:
print("label-layer cells:", len(lab), "| value agreement:", f"{lab.status.isin(['match', 'sign']).mean():.1%}")
print("fuzzy-layer cells:", int((cmp.mapping == "fuzzy").sum()),
      "| ambiguous:", int((cmp.mapping == "ambiguous").sum()), "| none:", int((cmp.mapping == "none").sum()))

label-layer cells: 73 | value agreement: 100.0%
fuzzy-layer cells: 0 | ambiguous: 0 | none: 0
